In [16]:
!pip install -q "streamlit>=1.38.0" streamlit-option-menu pyngrok pyjwt bcrypt plotly extra-streamlit-components


In [17]:
%%writefile otp_service.py

import os
import secrets
import smtplib
import datetime
import time
import jwt
import bcrypt
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# CRYPTO & HASHING HELPERS

def hash_txt(text: str) -> str:

    if not text:
        return ""
    return bcrypt.hashpw(text.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")

def check_txt(text: str, hashed: str) -> bool:

    if not text or not hashed:
        return False
    try:
        return bcrypt.checkpw(text.encode("utf-8"), hashed.encode("utf-8"))
    except Exception:
        return False
# CRYPTOGRAPHICALLY SECURE OTP GENERATION

def generate_otp() -> str:

    random_int = secrets.randbelow(1000000)
    return f"{random_int:06d}"

# JWT OTP TOKEN CREATION & VERIFICATION

def make_otp_token(email: str, otp: str, jwt_secret: str, expiry_minutes: int = 5) -> str:

    if not jwt_secret:
        raise ValueError("JWT signing secret must be provided.")

    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "sub": email.lower().strip(),
        "otp_hash": hash_txt(otp),
        "type": "password_reset_otp",
        "jti": secrets.token_hex(16),
        "iat": now,
        "exp": now + datetime.timedelta(minutes=expiry_minutes)
    }
    return jwt.encode(payload, jwt_secret, algorithm="HS256")


def verify_otp_token(
    token: str,
    input_otp: str,
    email: str,
    jwt_secret: str,
    expiry_minutes: int = 5
) -> tuple:

    if not token:
        return False, "Verification token missing. Please request a new OTP."

    cleaned_otp = input_otp.strip() if input_otp else ""
    if len(cleaned_otp) != 6 or not cleaned_otp.isdigit():
        return False, "Please enter a valid 6-digit numerical OTP code."

    try:
        payload = jwt.decode(token, jwt_secret, algorithms=["HS256"])

        # Validate subject email
        if payload.get("sub") != email.lower().strip():
            return False, "Security verification failed: Token does not match the active account."

        # Validate token purpose
        if payload.get("type") != "password_reset_otp":
            return False, "Invalid token type for password recovery."

        # Verify OTP hash
        stored_otp_hash = payload.get("otp_hash")
        if check_txt(cleaned_otp, stored_otp_hash):
            return True, "Valid"

        return False, "Invalid or incorrect 6-digit OTP code. Please check and try again."

    except jwt.ExpiredSignatureError:
        return False, f"⚠️This OTP code expired after {expiry_minutes} minutes. Please request a new code."
    except jwt.InvalidSignatureError:
        return False, "Security verification failed: Tampered or invalid token signature."
    except Exception:
        return False, "Invalid or corrupted verification session."


# RATE LIMITING & COOLDOWN HELPERS

def can_request_otp(last_sent_timestamp: float, cooldown_seconds: int = 60) -> tuple:

    if not last_sent_timestamp:
        return True, 0

    elapsed = time.time() - last_sent_timestamp
    if elapsed >= cooldown_seconds:
        return True, 0

    remaining = int(cooldown_seconds - elapsed) + 1
    return False, remaining

# PASSWORD HISTORY & REUSE PREVENTION (SIR'S CODE INTEGRATION)

def record_password_history(conn, email: str, password_hash: str) -> None:

    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    conn.execute(
        "INSERT INTO password_history (email, password_hash, set_at) VALUES (?, ?, ?)",
        (email.lower().strip(), password_hash, now_str)
    )
    conn.commit()


def check_password_reused(conn, email: str, new_password: str, history_limit: int = 5) -> bool:

    cursor = conn.cursor()
    cursor.execute(
        "SELECT password_hash FROM password_history WHERE email = ? ORDER BY id DESC LIMIT ?",
        (email.lower().strip(), history_limit)
    )
    rows = cursor.fetchall()
    for (old_hash,) in rows:
        if check_txt(new_password, old_hash):
            return True
    return False

# GMAIL SMTP MULTIPART EMAIL DELIVERY

def send_professional_email(
    sender_email: str,
    to_email: str,
    otp: str,
    app_pass: str,
    expiry_minutes: int = 5
) -> tuple:

    if not app_pass:
        return False, "Gmail App Password not found! Please configure EMAIL_PASSWORD in Colab Secrets."
    if not sender_email:
        return False, "Sender email address not configured. Please configure SENDER_EMAIL."

    msg = MIMEMultipart("alternative")
    msg["From"] = f"FreightQuote AI Verification <{sender_email}>"
    msg["To"] = to_email
    msg["Subject"] = "FreightQuote AI Verification - Password Reset Code"
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = sender_email

    # Plain-text version
    text_body = (
        f"FreightQuote AI Verification\n\n"
        f"We received a request to reset your password for {to_email}.\n\n"
        f"Your 6-Digit Verification Code is: {otp}\n\n"
        f"This code will expire in {expiry_minutes} minutes.\n"
        f"If you did not request this password reset, please ignore this email.\n\n"
        f"© 2026 FreightQuote AI. All rights reserved."
    )


    html_body = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <style>
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            background-color: #F5FAFF;
            margin: 0;
            padding: 24px;
            color: #475569;
        }}
        .container {{
            max-width: 520px;
            margin: 0 auto;
            background-color: #FFFFFF;
            border: 1px solid #E2E8F0;
            border-radius: 12px;
            padding: 32px 28px;
            text-align: center;
        }}
        .header {{
            font-size: 24px;
            font-weight: 700;
            color: #475569;
            margin-bottom: 12px;
        }}
        .subtext {{
            font-size: 14px;
            color: #64748B;
            line-height: 1.6;
            margin-bottom: 24px;
        }}
        .otp-container {{
            margin: 20px 0;
        }}
        .otp-box {{
            background-color: #E4F2FF;
            color: #475569;
            font-size: 32px;
            font-weight: 700;
            letter-spacing: 8px;
            padding: 14px 28px;
            border: 1px solid #8CCBFA;
            border-radius: 8px;
            display: inline-block;
        }}
        .timer {{
            font-size: 13px;
            color: #64748B;
            font-weight: 500;
            margin-top: 14px;
        }}
        .footer {{
            font-size: 12px;
            color: #94A3B8;
            margin-top: 28px;
            border-top: 1px solid #E2E8F0;
            padding-top: 16px;
            line-height: 1.5;
        }}
    </style>
</head>
<body>
    <div class="container">
       <div class="header">FreightQuote AI Verification</div>
        <div class="subtext">
            We received a request to reset the password for <b>{to_email}</b>.<br>
            Please use the 6-digit verification code below:
        </div>
        <div class="otp-container">
            <div class="otp-box">{otp}</div>
            <div class="timer">Valid for <b>{expiry_minutes} minutes</b></div>
        </div>
        <div class="footer">
            If you did not initiate this request, you can safely disregard this email.<br>
            &copy; 2026 FreightQuote AI. All rights reserved.
        </div>
    </div>
</body>
</html>"""

    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    server = None
    try:
        server = smtplib.SMTP("smtp.gmail.com", 587, timeout=15)
        server.starttls()
        server.login(sender_email, app_pass)
        server.sendmail(sender_email, to_email, msg.as_string())
        return True, "Email sent successfully!"
    except smtplib.SMTPAuthenticationError:
        return False, "SMTP Authentication Failed: Please verify your Gmail App Password."
    except Exception as e:
        return False, f"Failed to send email: {str(e)}"
    finally:
        if server:
            try:
                server.quit()
            except Exception:
                pass


Overwriting otp_service.py


In [18]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/FreightQuoteDB', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, re, subprocess, secrets
from contextlib import contextmanager
import streamlit as st
from streamlit_option_menu import option_menu
try:
    import extra_streamlit_components as stx
except ImportError:
    stx = None

# Import our comprehensive OTP service from otp_service.py
from otp_service import (
    hash_txt, check_txt, generate_otp, make_otp_token,
    verify_otp_token, send_professional_email,
    record_password_history, check_password_reused
)

# SETTINGS AT THE TOP OF app.py
LOGIN_FREE_ATTEMPTS = 3
LOGIN_LOCK_STEPS_SECONDS = [120, 300, 600]
OTP_FREE_ATTEMPTS = 3
OTP_LOCK_STEPS_SECONDS = [60, 180, 300, 600]
OTP_RESEND_WAIT_SECONDS = 60
DB_PATH = "/content/drive/MyDrive/FreightQuoteDB/infosys_portal.db"

# FORCE STREAMLIT LIGHT THEME CONFIGURATION
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write(
        '[theme]\n'
        'base="light"\n'
        'primaryColor="#8CCBFA"\n'
        'backgroundColor="#F5FAFF"\n'
        'secondaryBackgroundColor="#FFFFFF"\n'
        'textColor="#475569"\n'
    )

st.set_page_config(
    page_title="FreightQuote AI",
    layout="wide",
    initial_sidebar_state="expanded"
)

COLORS = {
    "bg_main": "#F5FAFF",
    "bg_sidebar": "#FFFFFF",
    "bg_card": "#FFFFFF",
    "bg_card_alt": "#E4F2FF",
    "text_main": "#475569",
    "text_heading": "#475569",
    "text_muted": "#64748B",
    "accent": "#8CCBFA",
    "accent_hover": "#E4F2FF",
    "accent_active": "#72BAF0",
    "accent_text": "#475569",
    "border": "#E2E8F0",
    "border_light": "#E2E8F0",
    "disabled": "#F1F5F9",
    "success": "#34D399",
    "danger": "#F87171"
}

def get_secret(key, default=""):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        return val if val else default
    except Exception:
        return os.environ.get(key, default)

JWT_SECRET = get_secret("JWT_SECRET", "FreightQuoteSecretKey2026!AI")
SENDER_EMAIL = get_secret("EMAIL_ID") or get_secret("SENDER_EMAIL", "tirth12222@gmail.com")
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD", None)
ADMIN_EMAIL = get_secret("ADMIN_EMAIL", "infosys@ai")
ADMIN_PASSWORD = get_secret("ADMIN_PASSWORD", "Admin@123")
OTP_EXPIRY_MINUTES = 5

st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

    html, body, .stApp {{
        background: {COLORS['bg_main']} !important;
        font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
        color: {COLORS['text_main']} !important;
    }}

    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* Sidebar toggle button */
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important;
        display: flex !important;
        opacity: 1 !important;
        background-color: {COLORS['accent']} !important;
        border: 1px solid {COLORS['border']} !important;
        border-radius: 6px !important;
        padding: 6px !important;
        margin: 8px !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important;
        color: {COLORS['text_heading']} !important;
        stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1080px; }}
    h1, h2, h3, h4 {{
        font-family: 'Inter', sans-serif !important;
        font-weight: 700 !important;
        color: {COLORS['text_heading']} !important;
        letter-spacing: -0.02em !important;
    }}

    div[data-baseweb="input"] {{
        background-color: {COLORS['bg_card']} !important;
        border: 1px solid {COLORS['border']} !important;
        border-radius: 8px !important;
        transition: all 0.2s ease !important;
    }}
    div[data-baseweb="input"]:focus-within {{
        border-color: {COLORS['accent']} !important;
        box-shadow: 0 0 0 3px {COLORS['accent_hover']} !important;
    }}
    div[data-baseweb="input"] input {{
        color: {COLORS['text_main']} !important;
        font-family: 'Inter', sans-serif !important;
        font-size: 14px !important;
    }}

    div[data-baseweb="select"] > div {{
        background-color: {COLORS['bg_card']} !important;
        border: 1px solid {COLORS['border']} !important;
        border-radius: 8px !important;
        transition: all 0.2s ease !important;
    }}
    div[data-baseweb="select"]:focus-within > div {{
        border-color: {COLORS['accent']} !important;
        box-shadow: 0 0 0 3px {COLORS['accent_hover']} !important;
    }}
    div[data-baseweb="select"] span {{
        color: {COLORS['text_main']} !important;
    }}
    ul[role="listbox"] {{
        background-color: {COLORS['bg_card']} !important;
        border: 1px solid {COLORS['border']} !important;
    }}
    li[role="option"] {{
        color: {COLORS['text_main']} !important;
        background-color: {COLORS['bg_card']} !important;
    }}
    li[role="option"]:hover, li[role="option"][aria-selected="true"] {{
        background-color: {COLORS['accent_hover']} !important;
        color: {COLORS['text_heading']} !important;
    }}

    /* Target both regular stButton and stFormSubmitButton */
    div[data-testid="stButton"] button,
    div[data-testid="stFormSubmitButton"] button {{
        background-color: {COLORS['accent']} !important;
        color: {COLORS['accent_text']} !important;
        border: 1px solid {COLORS['accent']} !important;
        border-radius: 8px !important;
        font-family: 'Inter', sans-serif !important;
        font-weight: 600 !important;
        font-size: 14px !important;
        height: 44px !important;
        min-height: 44px !important;
        white-space: nowrap !important;
        display: flex !important;
        align-items: center !important;
        justify-content: center !important;
        padding: 0px 16px !important;
        width: 100%;
        transition: all 0.18s ease !important;
        cursor: pointer !important;
        box-shadow: none !important;
    }}
    div[data-testid="stButton"] button:hover,
    div[data-testid="stFormSubmitButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important;
        border-color: {COLORS['accent']} !important;
        color: {COLORS['text_heading']} !important;
    }}
    div[data-testid="stButton"] button:active,
    div[data-testid="stFormSubmitButton"] button:active {{
        background-color: {COLORS['accent_active']} !important;
        transform: translateY(1px) !important;
    }}
    div[data-testid="stButton"] button:disabled,
    div[data-testid="stFormSubmitButton"] button:disabled {{
        background-color: {COLORS['disabled']} !important;
        border-color: {COLORS['border']} !important;
        color: {COLORS['text_muted']} !important;
        cursor: not-allowed !important;
    }}

    section[data-testid="stSidebar"] {{
        background: {COLORS['bg_sidebar']} !important;
        border-right: 1px solid {COLORS['border']} !important;
    }}
    .pn-card {{
        background: {COLORS['bg_card']};
        border: 1px solid {COLORS['border']};
        border-radius: 10px;
        padding: 24px;
        box-shadow: none;
    }}
    .account-row {{
        display: flex;
        justify-content: space-between;
        padding: 10px 0;
        border-bottom: 1px solid {COLORS['border']};
        font-size: 14px;
    }}
    .account-label {{
        color: {COLORS['text_muted']};
        font-weight: 500;
    }}
    .account-value {{
        color: {COLORS['text_main']};
        font-weight: 600;
    }}

    /* Ensure option_menu in sidebar is never hidden by zero-height iframe rules */
    section[data-testid="stSidebar"] div:has(iframe[title="streamlit_option_menu.option_menu"]),
    section[data-testid="stSidebar"] iframe[title="streamlit_option_menu.option_menu"],
    section[data-testid="stSidebar"] div[data-testid="stCustomComponentV1"] {{
        display: block !important;
        visibility: visible !important;
    }}
</style>
""", unsafe_allow_html=True)

# DATABASE UTILITIES (SHORT-LIVED CONNECTIONS)
def get_db():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def query_db(query, params=(), one=False):
    conn = get_db()
    try:
        cur = conn.cursor()
        cur.execute(query, params)
        rv = cur.fetchall()
        return (rv[0] if rv else None) if one else rv
    finally:
        conn.close()

def execute_db(query, params=()):
    conn = get_db()
    try:
        cur = conn.cursor()
        cur.execute(query, params)
        conn.commit()
    finally:
        conn.close()

def init_db():
    conn = get_db()
    try:
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT,
            role TEXT DEFAULT 'user',
            created_at TEXT
        )""")

        conn.execute("""CREATE TABLE IF NOT EXISTS password_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT,
            password_hash TEXT,
            set_at TEXT,
            FOREIGN KEY(email) REFERENCES users(email)
        )""")

        cols = [col[1] for col in conn.execute("PRAGMA table_info(users)").fetchall()]
        if "role" not in cols:
            conn.execute("ALTER TABLE users ADD COLUMN role TEXT DEFAULT 'user'")
        if "created_at" not in cols:
            conn.execute("ALTER TABLE users ADD COLUMN created_at TEXT")

        ph_cols = [col[1] for col in conn.execute("PRAGMA table_info(password_history)").fetchall()]
        if "set_at" not in ph_cols:
            conn.execute("ALTER TABLE password_history ADD COLUMN set_at TEXT")

        conn.execute("""CREATE TABLE IF NOT EXISTS login_attempts (
            identifier TEXT PRIMARY KEY,
            failed_attempts INTEGER DEFAULT 0,
            lock_level INTEGER DEFAULT 0,
            lock_until REAL DEFAULT 0,
            is_permanent INTEGER DEFAULT 0,
            updated_at TEXT
        )""")

        conn.execute("""CREATE TABLE IF NOT EXISTS recovery_attempts (
            email TEXT PRIMARY KEY,
            failed_attempts INTEGER DEFAULT 0,
            lock_level INTEGER DEFAULT 0,
            lock_until REAL DEFAULT 0,
            resend_until REAL DEFAULT 0,
            updated_at TEXT
        )""")

        conn.execute("""CREATE TABLE IF NOT EXISTS recovery_sessions (
            session_id TEXT PRIMARY KEY,
            email TEXT,
            method TEXT,
            step TEXT,
            otp_token TEXT,
            otp_verified INTEGER DEFAULT 0,
            verified_at REAL DEFAULT 0,
            expires_at REAL,
            created_at TEXT
        )""")

        # Username case-insensitive unique index with try/except
        try:
            conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_users_username_lower ON users (LOWER(username))")
        except Exception as e:
            print("Note: Could not create unique index on LOWER(username):", e)

        # Seed Administrator if not present
        admin_record = conn.execute("SELECT id FROM users WHERE email=? OR role='admin'", (ADMIN_EMAIL,)).fetchone()
        if not admin_record:
            now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            admin_h = hash_txt(ADMIN_PASSWORD)
            conn.execute(
                "INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role, created_at) "
                "VALUES (?, ?, ?, ?, ?, ?, ?)",
                ("Administrator", ADMIN_EMAIL, admin_h, "What is your pet name?", hash_txt("admin"), "admin", now_str)
            )
            record_password_history(conn, ADMIN_EMAIL, admin_h)
            conn.commit()
    finally:
        conn.close()

init_db()

# RECOVERY SESSION DATABASE HELPERS (CHANGE 6 FORGOT PASSWORD PERSISTENCE)
def get_recovery_session(session_id: str):
    if not session_id:
        return None
    row = query_db(
        "SELECT email, method, step, otp_token, otp_verified, verified_at, expires_at FROM recovery_sessions WHERE session_id=?",
        (session_id,),
        one=True
    )
    if not row:
        return None
    email, method, step, otp_token, otp_verified, verified_at, expires_at = row
    now = time.time()
    if expires_at and now > expires_at:
        delete_recovery_session(session_id)
        return None
    if otp_verified:
        # OTP-verified flag expires strictly after 10 minutes (600s)
        if now - verified_at > 600:
            delete_recovery_session(session_id)
            return None
    return {
        "session_id": session_id,
        "email": email,
        "method": method,
        "step": step,
        "otp_token": otp_token,
        "otp_verified": bool(otp_verified),
        "verified_at": verified_at
    }

def save_recovery_session(session_id: str, email: str, method: str, step: str, otp_token: str = None, otp_verified: bool = False):
    now = time.time()
    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    expires_at = now + 1800  # 30 min session limit
    verified_at = now if otp_verified else 0.0
    execute_db(
        """INSERT INTO recovery_sessions (session_id, email, method, step, otp_token, otp_verified, verified_at, expires_at, created_at)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
           ON CONFLICT(session_id) DO UPDATE SET
               email=excluded.email,
               method=excluded.method,
               step=excluded.step,
               otp_token=excluded.otp_token,
               otp_verified=excluded.otp_verified,
               verified_at=CASE WHEN excluded.otp_verified = 1 THEN excluded.verified_at ELSE recovery_sessions.verified_at END,
               expires_at=excluded.expires_at""",
        (session_id, email, method, step, otp_token, 1 if otp_verified else 0, verified_at, expires_at, now_str)
    )

def delete_recovery_session(session_id: str):
    if not session_id:
        return
    execute_db("DELETE FROM recovery_sessions WHERE session_id=?", (session_id,))

# INPUT VALIDATION UTILITIES
def validate_email(email_str):
    if not email_str or "@" not in email_str:
        return False, "Email address must contain '@'."
    parts = email_str.split("@")
    if len(parts) != 2:
        return False, "Invalid email address format."
    local_part, domain_part = parts[0], parts[1]
    if len(local_part) < 2:
        return False, "Email must have at least 2 characters before the '@' symbol."
    if "." not in domain_part:
        return False, "Email domain must contain a dot (.)."
    domain_parts = domain_part.rsplit(".", 1)
    domain_name, top_domain = domain_parts[0], domain_parts[1]
    if len(domain_name) < 2:
        return False, "Email must have at least 2 characters between '@' and the domain dot."
    if len(top_domain) < 2:
        return False, "Email must have at least 2 characters after the domain dot."
    email_pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
    if not re.match(email_pattern, email_str):
        return False, "Email contains invalid characters."
    return True, "Valid"

def validate_password(pwd):
    if len(pwd) < 8:
        return False, "Password must be at least 8 characters long."
    if not any(c.isupper() for c in pwd):
        return False, "Password must include at least one uppercase letter (A-Z)."
    if not any(c.islower() for c in pwd):
        return False, "Password must include at least one lowercase letter (a-z)."
    if not any(c.isdigit() for c in pwd):
        return False, "Password must include at least one digit (0-9)."
    special_chars = set("!@#$%^&*()_+-=[]{}|;':,./<>?`~\"")
    if not any(c in special_chars for c in pwd):
        return False, "Password must include at least one special character (!@#$%^&*)."
    return True, "Valid"

# JWT HELPERS
def make_jwt(email, role="user"):
    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "email": email,
        "role": role,
        "iat": now,
        "exp": now + datetime.timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None

# MONKEY-PATCH CookieManager.delete TO PREVENT KeyError: '<cookie>'
if stx is not None:
    def _safe_delete(self, cookie, key="delete"):
        if not cookie:
            return
        try:
            if not hasattr(self, "cookies") or not isinstance(self.cookies, dict):
                self.cookies = {}
            self.cookies.setdefault(cookie, "")
            self.cookie_manager(method="delete", cookie=cookie, key=key, default=False)
            self.cookies.pop(cookie, None)
        except Exception:
            pass
    stx.CookieManager.delete = _safe_delete

# INITIALIZE COOKIE MANAGER WITH FIXED KEY
cookie_manager = None
cookies = {}
if stx is not None:
    try:
        cookie_manager = stx.CookieManager(key="fq_cookie_manager")
        cookies = cookie_manager.get_all() or {}
    except Exception:
        cookie_manager = None
        cookies = {}

auth_jwt_cookie = cookies.get("fq_jwt") if cookies else None
rec_sid_cookie = cookies.get("fq_rec_sid") if cookies else None

def safe_delete_cookie(cookie_name, key=None):
    if not cookie_name or cookie_manager is None:
        return
    try:
        if hasattr(cookie_manager, "cookies") and isinstance(cookie_manager.cookies, dict):
            cookie_manager.cookies.setdefault(cookie_name, "")
        cookie_manager.delete(cookie_name, key=key or f"del_{cookie_name}_{int(time.time()*1000)}")
        if hasattr(cookie_manager, "cookies") and isinstance(cookie_manager.cookies, dict):
            cookie_manager.cookies.pop(cookie_name, None)
    except Exception:
        pass

def safe_set_cookie(cookie_name, val, key=None, expires_at=None):
    if not cookie_name or cookie_manager is None:
        return
    try:
        cookie_manager.set(
            cookie_name,
            val,
            key=key or f"set_{cookie_name}_{int(time.time()*1000)}",
            expires_at=expires_at or (datetime.datetime.now() + datetime.timedelta(hours=2))
        )
    except Exception:
        pass

# Check & verify auth token
# RULE (Bug 3): A missing cookie must never log out a user whose session already has a valid token.
# The cookie may only be used to restore a session when the session has no token.
is_authenticated = False
auth_email = None
auth_role = "user"

# 1. First check if session state already holds a valid token
if st.session_state.get("token"):
    payload = verify_jwt(st.session_state.token)
    if payload:
        c_email = payload.get("email")
        user_row = query_db("SELECT username, role FROM users WHERE email=?", (c_email,), one=True)
        if user_row:
            is_authenticated = True
            auth_email = c_email
            auth_role = user_row[1] if user_row[1] else "user"
        else:
            st.session_state.token = None
    else:
        st.session_state.token = None

# 2. If session state does NOT have a valid token, attempt to restore from cookie
if not is_authenticated and auth_jwt_cookie:
    payload = verify_jwt(auth_jwt_cookie)
    if payload:
        c_email = payload.get("email")
        user_row = query_db("SELECT username, role FROM users WHERE email=?", (c_email,), one=True)
        if user_row:
            st.session_state.token = auth_jwt_cookie
            is_authenticated = True
            auth_email = c_email
            auth_role = user_row[1] if user_row[1] else "user"
        else:
            safe_delete_cookie("fq_jwt", key="del_dead_jwt")
    else:
        safe_delete_cookie("fq_jwt", key="del_expired_jwt")

# URL QUERY PARAMETER NAVIGATION (CHANGE 6)
raw_page = st.query_params.get("page", None)
valid_pages = ["Login", "Signup", "Forgot", "Dashboard"]

if is_authenticated:
    # A logged-in user who opens a login-type page goes to the Dashboard
    if raw_page in ["Login", "Signup", "Forgot"] or not raw_page or raw_page not in valid_pages:
        st.session_state.page = "Dashboard"
        st.query_params["page"] = "Dashboard"
    else:
        st.session_state.page = raw_page
else:
    # A logged-out user who asks for the Dashboard goes to Login
    if raw_page == "Dashboard" or not raw_page or raw_page not in valid_pages:
        st.session_state.page = "Login"
        st.query_params["page"] = "Login"
    else:
        st.session_state.page = raw_page

# SESSION STATE DEFAULTS
for key, val in [
    ("reset_email", None),
    ("reset_mode", None),
    ("sq_p", None),
    ("otp_token", None),
    ("otp_verified", False),
    ("otp_stage", "email"),
    ("rec_sid", None),
    ("signup_errors", {}),
    ("last_login_key", "")
]:
    if key not in st.session_state:
        st.session_state[key] = val

# RESTORE FORGOT PASSWORD STEP FROM DATABASE (CHANGE 6)
# RULE (Bug 2): Restore Forgot Password step only when restoring a refreshed session
# (i.e. when session state has no rec_sid and cookie exists).
# NEVER overwrite a newer session state step with an older saved one.
if st.session_state.page == "Forgot":
    if not st.session_state.rec_sid and rec_sid_cookie:
        rec_session = get_recovery_session(rec_sid_cookie)
        if rec_session:
            st.session_state.rec_sid = rec_sid_cookie
            st.session_state.reset_email = rec_session["email"]
            st.session_state.reset_mode = rec_session["method"]
            st.session_state.otp_stage = rec_session["step"]
            st.session_state.otp_token = rec_session["otp_token"]
            st.session_state.otp_verified = rec_session["otp_verified"]
            if rec_session["method"] == "sq":
                u_sq = query_db("SELECT security_question FROM users WHERE email=?", (rec_session["email"],), one=True)
                st.session_state.sq_p = u_sq[0] if u_sq else None
        else:
            safe_delete_cookie("fq_rec_sid", key="del_expired_sid")
    elif st.session_state.rec_sid:
        # Verify active session has not expired in database
        rec_session = get_recovery_session(st.session_state.rec_sid)
        if not rec_session:
            clear_reset_state()

def navigate(p):
    st.session_state.page = p
    st.query_params["page"] = p
    st.rerun()

def clear_reset_state():
    curr_sid = st.session_state.get("rec_sid")
    if curr_sid:
        try:
            delete_recovery_session(curr_sid)
        except Exception:
            pass
        safe_delete_cookie("fq_rec_sid", key="del_reset_sid")
    st.session_state.rec_sid = None
    st.session_state.reset_email = None
    st.session_state.reset_mode = None
    st.session_state.sq_p = None
    st.session_state.otp_token = None
    st.session_state.otp_verified = False
    st.session_state.otp_stage = "email"

def auth_header(title, sub="Intelligent Authentication Portal"):
    st.markdown(f"""
    <div style="width:100%;text-align:center;padding:1.5rem 0 1rem;box-sizing:border-box;">
        <h1 style="font-size:1.85rem !important;margin:0;color:{COLORS['text_heading']};font-weight:700;">FreightQuote AI</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;">
        <span style="font-size:1.05rem;font-weight:600;color:{COLORS['text_main']};">{title}</span>
    </div>
    """, unsafe_allow_html=True)

# LOCKOUT & ATTEMPTS HELPERS
def resolve_login_key(typed_text: str) -> tuple:
    clean = typed_text.strip()
    if not clean:
        return "", False
    row = query_db(
        "SELECT email, role FROM users WHERE LOWER(email)=LOWER(?) OR LOWER(username)=LOWER(?)",
        (clean, clean),
        one=True
    )
    if row:
        email, role = row[0].lower(), row[1]
        is_admin = (role == "admin" or email == ADMIN_EMAIL.lower())
        return email, is_admin
    else:
        is_admin = (clean.lower() == ADMIN_EMAIL.lower() or clean.lower() == "administrator")
        return clean.lower(), is_admin

def get_login_lock_status(key: str) -> dict:
    if not key:
        return {"is_blocked": False, "is_locked": False, "remaining_seconds": 0, "failed_attempts": 0, "lock_level": 0}
    row = query_db(
        "SELECT failed_attempts, lock_level, lock_until, is_permanent FROM login_attempts WHERE identifier=?",
        (key,),
        one=True
    )
    if not row:
        return {"is_blocked": False, "is_locked": False, "remaining_seconds": 0, "failed_attempts": 0, "lock_level": 0}
    failed_attempts, lock_level, lock_until, is_permanent = row
    now = time.time()
    if is_permanent:
        return {"is_blocked": True, "is_locked": True, "remaining_seconds": 0, "failed_attempts": failed_attempts, "lock_level": lock_level}
    remaining = max(0, int(lock_until - now)) if lock_until else 0
    return {
        "is_blocked": False,
        "is_locked": remaining > 0,
        "remaining_seconds": remaining,
        "failed_attempts": failed_attempts,
        "lock_level": lock_level
    }

def record_login_failure(key: str, is_admin: bool):
    now = time.time()
    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    row = query_db(
        "SELECT failed_attempts, lock_level, lock_until, is_permanent FROM login_attempts WHERE identifier=?",
        (key,),
        one=True
    )
    if not row:
        failed = 1
        level = 0
        lock_until = 0.0
        is_perm = 0
    else:
        failed, level, lock_until, is_perm = row
        if is_perm:
            return
        failed += 1

    if failed >= LOGIN_FREE_ATTEMPTS:
        if is_admin:
            step_idx = min(level, len(LOGIN_LOCK_STEPS_SECONDS) - 1)
            duration = LOGIN_LOCK_STEPS_SECONDS[step_idx]
            lock_until = now + duration
            level = min(level + 1, len(LOGIN_LOCK_STEPS_SECONDS) - 1)
            is_perm = 0
            failed = 0
        else:
            if level < len(LOGIN_LOCK_STEPS_SECONDS):
                duration = LOGIN_LOCK_STEPS_SECONDS[level]
                lock_until = now + duration
                level += 1
                failed = 0
                is_perm = 0
            else:
                is_perm = 1
                lock_until = 0.0

    execute_db(
        """INSERT INTO login_attempts (identifier, failed_attempts, lock_level, lock_until, is_permanent, updated_at)
           VALUES (?, ?, ?, ?, ?, ?)
           ON CONFLICT(identifier) DO UPDATE SET
               failed_attempts=excluded.failed_attempts,
               lock_level=excluded.lock_level,
               lock_until=excluded.lock_until,
               is_permanent=excluded.is_permanent,
               updated_at=excluded.updated_at""",
        (key, failed, level, lock_until, is_perm, now_str)
    )

def reset_login_attempts(key: str):
    if not key:
        return
    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    execute_db(
        """INSERT INTO login_attempts (identifier, failed_attempts, lock_level, lock_until, is_permanent, updated_at)
           VALUES (?, 0, 0, 0, 0, ?)
           ON CONFLICT(identifier) DO UPDATE SET
               failed_attempts=0,
               lock_level=0,
               lock_until=0,
               is_permanent=0,
               updated_at=excluded.updated_at""",
        (key, now_str)
    )

def get_recovery_status(email: str) -> dict:
    clean_email = email.lower().strip()
    if not clean_email:
        return {"is_locked": False, "remaining_seconds": 0, "resend_remaining": 0, "failed_attempts": 0, "lock_level": 0}
    row = query_db(
        "SELECT failed_attempts, lock_level, lock_until, resend_until FROM recovery_attempts WHERE email=?",
        (clean_email,),
        one=True
    )
    if not row:
        return {"is_locked": False, "remaining_seconds": 0, "resend_remaining": 0, "failed_attempts": 0, "lock_level": 0}
    failed_attempts, lock_level, lock_until, resend_until = row
    now = time.time()
    lock_rem = max(0, int(lock_until - now)) if lock_until else 0
    resend_rem = max(0, int(resend_until - now)) if resend_until else 0
    return {
        "is_locked": lock_rem > 0,
        "remaining_seconds": lock_rem,
        "resend_remaining": resend_rem,
        "failed_attempts": failed_attempts,
        "lock_level": lock_level
    }

def record_recovery_failure(email: str):
    clean_email = email.lower().strip()
    now = time.time()
    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    row = query_db(
        "SELECT failed_attempts, lock_level, lock_until, resend_until FROM recovery_attempts WHERE email=?",
        (clean_email,),
        one=True
    )
    if not row:
        failed = 1
        level = 0
        lock_until = 0.0
        resend_until = 0.0
    else:
        failed, level, lock_until, resend_until = row
        failed += 1

    if failed >= OTP_FREE_ATTEMPTS:
        step_idx = min(level, len(OTP_LOCK_STEPS_SECONDS) - 1)
        duration = OTP_LOCK_STEPS_SECONDS[step_idx]
        lock_until = now + duration
        level = min(level + 1, len(OTP_LOCK_STEPS_SECONDS) - 1)
        failed = 0

    execute_db(
        """INSERT INTO recovery_attempts (email, failed_attempts, lock_level, lock_until, resend_until, updated_at)
           VALUES (?, ?, ?, ?, ?, ?)
           ON CONFLICT(email) DO UPDATE SET
               failed_attempts=excluded.failed_attempts,
               lock_level=excluded.lock_level,
               lock_until=excluded.lock_until,
               updated_at=excluded.updated_at""",
        (clean_email, failed, level, lock_until, now_str)
    )

def set_otp_resend_cooldown(email: str):
    clean_email = email.lower().strip()
    now = time.time()
    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    resend_until = now + OTP_RESEND_WAIT_SECONDS
    execute_db(
        """INSERT INTO recovery_attempts (email, failed_attempts, lock_level, lock_until, resend_until, updated_at)
           VALUES (?, 0, 0, 0, ?, ?)
           ON CONFLICT(email) DO UPDATE SET
               resend_until=excluded.resend_until,
               updated_at=excluded.updated_at""",
        (clean_email, resend_until, now_str)
    )

def reset_recovery_attempts(email: str):
    clean_email = email.lower().strip()
    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    execute_db(
        """INSERT INTO recovery_attempts (email, failed_attempts, lock_level, lock_until, resend_until, updated_at)
           VALUES (?, 0, 0, 0, 0, ?)
           ON CONFLICT(email) DO UPDATE SET
               failed_attempts=0,
               lock_level=0,
               lock_until=0,
               updated_at=excluded.updated_at""",
        (clean_email, now_str)
    )

def admin_unblock_user(email: str):
    clean_email = email.lower().strip()
    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    execute_db(
        """INSERT INTO login_attempts (identifier, failed_attempts, lock_level, lock_until, is_permanent, updated_at)
           VALUES (?, 0, 0, 0, 0, ?)
           ON CONFLICT(identifier) DO UPDATE SET
               failed_attempts=0,
               lock_level=0,
               lock_until=0,
               is_permanent=0,
               updated_at=excluded.updated_at""",
        (clean_email, now_str)
    )
    execute_db(
        """INSERT INTO recovery_attempts (email, failed_attempts, lock_level, lock_until, resend_until, updated_at)
           VALUES (?, 0, 0, 0, 0, ?)
           ON CONFLICT(email) DO UPDATE SET
               failed_attempts=0,
               lock_level=0,
               lock_until=0,
               resend_until=0,
               updated_at=excluded.updated_at""",
        (clean_email, now_str)
    )

# LIVE COUNTDOWN FRAGMENTS
@st.fragment(run_every=1)
def render_lock_countdown(target_time: float, message_prefix: str = "Too many failed attempts. Try again in "):
    remaining = max(0, int(target_time - time.time()))
    if remaining > 0:
        mins = remaining // 60
        secs = remaining % 60
        st.error(f"{message_prefix}{mins:02d}:{secs:02d}")
    else:
        st.rerun()

@st.fragment(run_every=1)
def render_resend_countdown(target_time: float):
    remaining = max(0, int(target_time - time.time()))
    if remaining > 0:
        mins = remaining // 60
        secs = remaining % 60
        st.warning(f"Please wait {mins:02d}:{secs:02d} before requesting a new code.")
    else:
        st.rerun()

# PAGE ROUTING
if not is_authenticated:
    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        # LOGIN PAGE
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")

            if st.session_state.get("signup_success_msg"):
                st.success(st.session_state.pop("signup_success_msg"))
            if st.session_state.get("login_info_msg"):
                st.success(st.session_state.pop("login_info_msg"))

            # Check lock status based on last entered / session identifier (kept outside form)
            resolved_key = st.session_state.get("last_login_key", "")
            login_status = get_login_lock_status(resolved_key) if resolved_key else {
                "is_blocked": False, "is_locked": False, "remaining_seconds": 0
            }

            if login_status["is_blocked"]:
                st.error("Your account has been blocked. Please contact the administrator.")
            elif login_status["is_locked"]:
                lock_row = query_db("SELECT lock_until FROM login_attempts WHERE identifier=?", (resolved_key,), one=True)
                if lock_row and lock_row[0]:
                    render_lock_countdown(lock_row[0], "Too many failed attempts. Try again in ")

            is_sign_in_disabled = login_status["is_blocked"] or login_status["is_locked"]

            # Wrap in st.form(..., border=False) to ensure inputs are recognized without Enter
            with st.form("login_form", border=False):
                login_id = st.text_input("Username or Email", placeholder="Enter your username or email").strip()
                pwd = st.text_input("Password", type="password", placeholder="Enter your password")
                st.markdown("<br>", unsafe_allow_html=True)

                col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
                sign_in_submitted = col_l.form_submit_button("Sign In", use_container_width=True, disabled=is_sign_in_disabled)
                create_acct_clicked = col_c.form_submit_button("Create Account", use_container_width=True)
                forgot_pwd_clicked = col_r.form_submit_button("Forgot Password", use_container_width=True)

            if create_acct_clicked:
                st.session_state.signup_errors = {}
                navigate("Signup")

            elif forgot_pwd_clicked:
                clear_reset_state()
                navigate("Forgot")

            elif sign_in_submitted:
                curr_key, is_admin_acct = resolve_login_key(login_id) if login_id else ("", False)
                st.session_state.last_login_key = curr_key

                if not login_id or not pwd:
                    st.error("Both username/email and password fields are required.")
                else:
                    curr_lock = get_login_lock_status(curr_key)
                    if curr_lock["is_blocked"]:
                        st.error("Your account has been blocked. Please contact the administrator.")
                    elif curr_lock["is_locked"]:
                        st.error("Too many failed attempts. Try again later.")
                    else:
                        r = query_db(
                            "SELECT email, password_hash, role FROM users WHERE email=? OR LOWER(username)=LOWER(?)",
                            (login_id.lower(), login_id),
                            one=True
                        )
                        if r and check_txt(pwd, r[1]):
                            user_email = r[0]
                            user_role = r[2] if len(r) > 2 and r[2] else "user"
                            token = make_jwt(user_email, user_role)
                            reset_login_attempts(curr_key)
                            safe_set_cookie(
                                "fq_jwt",
                                token,
                                key="cookie_set_jwt",
                                expires_at=datetime.datetime.now() + datetime.timedelta(hours=2)
                            )
                            st.session_state.token = token
                            st.session_state.page = "Dashboard"
                            st.query_params["page"] = "Dashboard"
                            st.rerun()
                        else:
                            record_login_failure(curr_key, is_admin_acct)
                            new_lock = get_login_lock_status(curr_key)
                            if new_lock["is_blocked"]:
                                st.error("Your account has been blocked. Please contact the administrator.")
                            elif new_lock["is_locked"]:
                                st.rerun()
                            else:
                                st.error("Invalid credentials. Please check your username/email and password.")

        # SIGNUP PAGE
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Your intelligent maritime freight platform")
            st.markdown(f"""
            <div style="font-size:13px; color:{COLORS['text_muted']}; margin-top:-6px; margin-bottom:14px;">
                <span style="color:#e11d48; font-weight:600;">*</span> All fields marked with an asterisk are required.
            </div>
            """, unsafe_allow_html=True)

            signup_errs = st.session_state.get("signup_errors", {})

            with st.form("signup_form", border=False):
                uname = st.text_input("Username *", key="signup_username", placeholder="e.g. john_doe")
                if "uname" in signup_errs:
                    st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["uname"]}</div>', unsafe_allow_html=True)

                email = st.text_input("Email address *", key="signup_email", placeholder="e.g. ab@cd.ef").lower().strip()
                if "email" in signup_errs:
                    st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["email"]}</div>', unsafe_allow_html=True)

                pwd = st.text_input("Password *", key="signup_pwd", type="password", placeholder="Min. 8 chars (upper, lower, digit, symbol)")
                if "pwd" in signup_errs:
                    st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["pwd"]}</div>', unsafe_allow_html=True)

                confirm_pwd = st.text_input("Confirm password *", key="signup_confirm_pwd", type="password", placeholder="Re-enter password")
                if "confirm_pwd" in signup_errs:
                    st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["confirm_pwd"]}</div>', unsafe_allow_html=True)

                sq_options = [
                    "Select a security question...",
                    "What is your pet name?",
                    "What is your mother's maiden name?",
                    "What is your favourite city?"
                ]
                sq = st.selectbox("Security Question *", sq_options, key="signup_sq")
                if "sq" in signup_errs:
                    st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["sq"]}</div>', unsafe_allow_html=True)

                sa = st.text_input("Security answer *", key="signup_sa", placeholder="Your answer")
                st.markdown(f'<div style="font-size:12px; color:{COLORS["text_muted"]}; margin-top:-6px; margin-bottom:8px;">The answer is case sensitive and must have no spaces.</div>', unsafe_allow_html=True)
                if "sa" in signup_errs:
                    st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["sa"]}</div>', unsafe_allow_html=True)

                st.markdown("<br>", unsafe_allow_html=True)

                signup_submitted = st.form_submit_button("Sign Up", use_container_width=True)

            if signup_submitted:
                new_errors = {}
                if not uname:
                    new_errors["uname"] = "Please enter your username."
                if not email:
                    new_errors["email"] = "Please enter your email address."
                if not pwd:
                    new_errors["pwd"] = "Please enter your password."
                if not confirm_pwd:
                    new_errors["confirm_pwd"] = "Please confirm your password."
                if sq == "Select a security question...":
                    new_errors["sq"] = "Please select a security question."
                if not sa:
                    new_errors["sa"] = "Please provide an answer to the security question."

                if new_errors:
                    st.session_state.signup_errors = new_errors
                    st.rerun()
                elif any(c.isspace() for c in sa):
                    st.error("Security answer must be a single word without spaces.")
                elif "@" in uname:
                    st.error("Username cannot contain '@'.")
                else:
                    is_valid_email, email_msg = validate_email(email)
                    if not is_valid_email:
                        st.error(f"Invalid Email: {email_msg}")
                    else:
                        is_valid_pwd, pwd_msg = validate_password(pwd)
                        if not is_valid_pwd:
                            st.error(f"Password Error: {pwd_msg}")
                        elif pwd != confirm_pwd:
                            st.error("Passwords do not match.")
                        elif email == ADMIN_EMAIL.lower() or uname.lower() == "administrator":
                            st.error("This username or email is reserved.")
                        else:
                            existing_u = query_db("SELECT id FROM users WHERE LOWER(username)=LOWER(?)", (uname,), one=True)
                            if existing_u:
                                st.error("Username already registered.")
                            else:
                                try:
                                    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                                    new_hash = hash_txt(pwd)
                                    conn = get_db()
                                    try:
                                        conn.execute(
                                            "INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role, created_at) "
                                            "VALUES (?, ?, ?, ?, ?, ?, ?)",
                                            (uname, email, new_hash, sq, hash_txt(sa), "user", now_str)
                                        )
                                        record_password_history(conn, email, new_hash)
                                        conn.commit()
                                    finally:
                                        conn.close()
                                    # Clear form inputs from session state so form doesn't stay filled
                                    for k in ["signup_username", "signup_email", "signup_pwd", "signup_confirm_pwd", "signup_sq", "signup_sa"]:
                                        st.session_state.pop(k, None)
                                    st.session_state.signup_errors = {}
                                    st.session_state.signup_success_msg = "Account created successfully. Please sign in with your credentials."
                                    time.sleep(1)
                                    navigate("Login")
                                except sqlite3.IntegrityError:
                                    st.error("Username or Email address already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("Back to Sign In", use_container_width=True):
                st.session_state.signup_errors = {}
                for k in ["signup_username", "signup_email", "signup_pwd", "signup_confirm_pwd", "signup_sq", "signup_sa"]:
                    st.session_state.pop(k, None)
                navigate("Login")

        # FORGOT PASSWORD
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            # Selection Step (Choose Method)
            if not st.session_state.reset_mode:
                with st.form("forgot_step1_form", border=False):
                    recovery_id = st.text_input("Registered Username or Email address", placeholder="you@infosys.com or username").strip()
                    st.markdown("<br>", unsafe_allow_html=True)

                    col_sq, col_otp = st.columns(2)
                    via_sq_clicked = col_sq.form_submit_button("Via Security Question", use_container_width=True)
                    via_otp_clicked = col_otp.form_submit_button("Via Email OTP", use_container_width=True)

                if via_sq_clicked:
                    if not recovery_id:
                        st.error("Please enter your registered username or email.")
                    else:
                        r = query_db(
                            "SELECT email, security_question FROM users WHERE email=? OR LOWER(username)=LOWER(?)",
                            (recovery_id.lower(), recovery_id),
                            one=True
                        )
                        if r:
                            t_email = r[0]
                            l_lock = get_login_lock_status(t_email)
                            if l_lock["is_blocked"]:
                                st.error("Your account has been blocked. Please contact the administrator.")
                            elif l_lock["is_locked"]:
                                st.error("Too many failed attempts. Try again later.")
                            else:
                                new_sid = secrets.token_hex(16)
                                save_recovery_session(new_sid, t_email, "sq", "sq")
                                safe_set_cookie("fq_rec_sid", new_sid, key="set_sid_sq", expires_at=datetime.datetime.now() + datetime.timedelta(hours=1))
                                st.session_state.rec_sid = new_sid
                                st.session_state.reset_email = t_email
                                st.session_state.sq_p = r[1]
                                st.session_state.reset_mode = "sq"
                                st.rerun()
                        else:
                            st.error("Invalid credentials. Please verify your username or email.")

                elif via_otp_clicked:
                    if not recovery_id:
                        st.error("Please enter your registered username or email address.")
                    else:
                        r = query_db(
                            "SELECT email FROM users WHERE email=? OR LOWER(username)=LOWER(?)",
                            (recovery_id.lower(), recovery_id),
                            one=True
                        )
                        if r:
                            t_email = r[0]
                            l_lock = get_login_lock_status(t_email)
                            if l_lock["is_blocked"]:
                                st.error("Your account has been blocked. Please contact the administrator.")
                            elif l_lock["is_locked"]:
                                st.error("Too many failed attempts. Try again later.")
                            else:
                                new_sid = secrets.token_hex(16)
                                save_recovery_session(new_sid, t_email, "otp", "email")
                                safe_set_cookie("fq_rec_sid", new_sid, key="set_sid_otp", expires_at=datetime.datetime.now() + datetime.timedelta(hours=1))
                                st.session_state.rec_sid = new_sid
                                st.session_state.reset_email = t_email
                                st.session_state.reset_mode = "otp"
                                st.session_state.otp_stage = "email"
                                st.rerun()
                        else:
                            st.error("Invalid credentials. Please verify your username or email.")

                st.markdown("<br>", unsafe_allow_html=True)
                if st.button("Back to Sign In", use_container_width=True):
                    clear_reset_state()
                    navigate("Login")

            # Security Question Recovery
            elif st.session_state.reset_mode == "sq":
                target_email = st.session_state.reset_email
                l_lock = get_login_lock_status(target_email)
                rec_stat = get_recovery_status(target_email)

                # Keep countdown timers outside form
                if l_lock["is_blocked"]:
                    st.error("Your account has been blocked. Please contact the administrator.")
                elif l_lock["is_locked"]:
                    l_row = query_db("SELECT lock_until FROM login_attempts WHERE identifier=?", (target_email,), one=True)
                    if l_row and l_row[0]:
                        render_lock_countdown(l_row[0], "Too many failed attempts. Try again in ")
                elif rec_stat["is_locked"]:
                    r_row = query_db("SELECT lock_until FROM recovery_attempts WHERE email=?", (target_email,), one=True)
                    if r_row and r_row[0]:
                        render_lock_countdown(r_row[0], "Too many failed attempts. Try again in ")

                is_sq_disabled = l_lock["is_blocked"] or l_lock["is_locked"] or rec_stat["is_locked"]

                with st.form("sq_form", border=False):
                    st.info(f"Security Question: {st.session_state.sq_p}")
                    ans = st.text_input("Your answer", placeholder="Security answer")
                    npw = st.text_input("New password", type="password", placeholder="Min. 8 chars (upper, lower, digit, symbol)")
                    confirm_npw = st.text_input("Confirm new password", type="password", placeholder="Re-enter new password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    sq_submitted = st.form_submit_button("Reset Password", use_container_width=True, disabled=is_sq_disabled)

                if sq_submitted:
                    curr_l = get_login_lock_status(target_email)
                    curr_r = get_recovery_status(target_email)
                    if curr_l["is_blocked"] or curr_l["is_locked"] or curr_r["is_locked"]:
                        st.error("Action not permitted while recovery is locked.")
                    elif not ans:
                        st.error("Please provide the answer to your security question.")
                    elif any(c.isspace() for c in ans):
                        st.error("Answer must be one word with no spaces.")
                    else:
                        is_valid_pwd, pwd_msg = validate_password(npw)
                        if not is_valid_pwd:
                            st.error(f"Password Error: {pwd_msg}")
                        elif npw != confirm_npw:
                            st.error("Passwords do not match.")
                        else:
                            r = query_db(
                                "SELECT security_answer_hash FROM users WHERE email=?",
                                (target_email,),
                                one=True
                            )
                            ans_clean = ans.strip()
                            if r and (check_txt(ans_clean, r[0]) or check_txt(ans_clean.lower(), r[0])):
                                conn = get_db()
                                try:
                                    if check_password_reused(conn, target_email, npw):
                                        st.error("For security, you cannot reuse your current or recent password.")
                                    else:
                                        new_h = hash_txt(npw)
                                        conn.execute("UPDATE users SET password_hash=? WHERE email=?", (new_h, target_email))
                                        record_password_history(conn, target_email, new_h)
                                        conn.commit()
                                        reset_recovery_attempts(target_email)
                                        clear_reset_state()
                                        st.session_state.login_info_msg = "Password updated successfully. Please sign in with your new password."
                                        time.sleep(1)
                                        navigate("Login")
                                finally:
                                    conn.close()
                            else:
                                record_recovery_failure(target_email)
                                new_r = get_recovery_status(target_email)
                                if new_r["is_locked"]:
                                    st.rerun()
                                else:
                                    st.error("Incorrect security answer.")

                st.markdown("<br>", unsafe_allow_html=True)
                if st.button("Cancel Recovery", use_container_width=True):
                    clear_reset_state()
                    navigate("Login")

            # Email OTP Recovery
            elif st.session_state.reset_mode == "otp":
                target_email = st.session_state.reset_email
                l_lock = get_login_lock_status(target_email)
                rec_stat = get_recovery_status(target_email)

                # Request OTP
                if st.session_state.otp_stage == "email":
                    if l_lock["is_blocked"]:
                        st.error("Your account has been blocked. Please contact the administrator.")
                    elif l_lock["is_locked"]:
                        l_row = query_db("SELECT lock_until FROM login_attempts WHERE identifier=?", (target_email,), one=True)
                        if l_row and l_row[0]:
                            render_lock_countdown(l_row[0], "Too many failed attempts. Try again in ")
                    elif rec_stat["is_locked"]:
                        r_row = query_db("SELECT lock_until FROM recovery_attempts WHERE email=?", (target_email,), one=True)
                        if r_row and r_row[0]:
                            render_lock_countdown(r_row[0], "Too many failed attempts. Try again in ")
                    elif rec_stat["resend_remaining"] > 0:
                        res_row = query_db("SELECT resend_until FROM recovery_attempts WHERE email=?", (target_email,), one=True)
                        if res_row and res_row[0]:
                            render_resend_countdown(res_row[0])

                    can_send = not (l_lock["is_blocked"] or l_lock["is_locked"] or rec_stat["is_locked"] or rec_stat["resend_remaining"] > 0)

                    st.info("A 6-digit verification code will be sent to the registered email address on file.")

                    if st.button("Send Verification Code", use_container_width=True, disabled=not can_send):
                        curr_l = get_login_lock_status(target_email)
                        curr_r = get_recovery_status(target_email)
                        if curr_l["is_blocked"] or curr_l["is_locked"] or curr_r["is_locked"]:
                            st.error("Action not permitted while recovery is locked.")
                        elif curr_r["resend_remaining"] > 0:
                            st.error("Please wait before requesting a new code.")
                        elif not EMAIL_PASSWORD:
                            st.error("Gmail App Password not found. Please configure EMAIL_PASSWORD in Colab Secrets.")
                        else:
                            otp = generate_otp()
                            with st.spinner("Sending verification code..."):
                                ok, msg = send_professional_email(SENDER_EMAIL, target_email, otp, EMAIL_PASSWORD, OTP_EXPIRY_MINUTES)
                            if ok:
                                otp_tok = make_otp_token(target_email, otp, JWT_SECRET, OTP_EXPIRY_MINUTES)
                                set_otp_resend_cooldown(target_email)
                                curr_sid = st.session_state.get("rec_sid") or secrets.token_hex(16)
                                save_recovery_session(curr_sid, target_email, "otp", "otp", otp_token=otp_tok, otp_verified=False)
                                safe_set_cookie("fq_rec_sid", curr_sid, key="set_sid_otp_entry", expires_at=datetime.datetime.now() + datetime.timedelta(hours=1))
                                st.session_state.rec_sid = curr_sid
                                st.session_state.otp_token = otp_tok
                                st.session_state.otp_stage = "otp"
                                st.success("Verification code sent to your registered email address. Please check your inbox.")
                                time.sleep(1)
                                st.rerun()
                            else:
                                st.error(f"Failed to send email: {msg}")

                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Cancel Recovery", use_container_width=True):
                        clear_reset_state()
                        navigate("Login")

                # Enter & Verify OTP
                elif st.session_state.otp_stage == "otp":
                    if l_lock["is_blocked"]:
                        st.error("Your account has been blocked. Please contact the administrator.")
                        st.session_state.otp_token = None
                    elif l_lock["is_locked"]:
                        l_row = query_db("SELECT lock_until FROM login_attempts WHERE identifier=?", (target_email,), one=True)
                        if l_row and l_row[0]:
                            render_lock_countdown(l_row[0], "Too many failed attempts. Try again in ")
                        st.session_state.otp_token = None
                    elif rec_stat["is_locked"]:
                        r_row = query_db("SELECT lock_until FROM recovery_attempts WHERE email=?", (target_email,), one=True)
                        if r_row and r_row[0]:
                            render_lock_countdown(r_row[0], "Too many failed attempts. Try again in ")
                        st.session_state.otp_token = None
                    elif rec_stat["resend_remaining"] > 0:
                        res_row = query_db("SELECT resend_until FROM recovery_attempts WHERE email=?", (target_email,), one=True)
                        if res_row and res_row[0]:
                            render_resend_countdown(res_row[0])

                    can_verify = not (l_lock["is_blocked"] or l_lock["is_locked"] or rec_stat["is_locked"])
                    can_resend = not (l_lock["is_blocked"] or l_lock["is_locked"] or rec_stat["is_locked"] or rec_stat["resend_remaining"] > 0)

                    with st.form("otp_entry_form", border=False):
                        st.info(f"Verification code sent. Valid for {OTP_EXPIRY_MINUTES} minutes.")
                        otp_input = st.text_input("Enter 6-Digit Verification Code", placeholder="e.g. 849201", max_chars=6).strip()
                        st.markdown("<br>", unsafe_allow_html=True)

                        c1, c2 = st.columns([1.2, 1])
                        verify_submitted = c1.form_submit_button("Verify Code", use_container_width=True, disabled=not can_verify)
                        resend_clicked = c2.form_submit_button("Resend / Change", use_container_width=True, disabled=not can_resend)

                    if resend_clicked:
                        curr_r = get_recovery_status(target_email)
                        if curr_r["resend_remaining"] > 0:
                            st.error("Please wait before requesting a new code.")
                        else:
                            curr_sid = st.session_state.get("rec_sid") or secrets.token_hex(16)
                            save_recovery_session(curr_sid, target_email, "otp", "email", otp_token=None, otp_verified=False)
                            st.session_state.otp_stage = "email"
                            st.session_state.otp_token = None
                            st.rerun()

                    elif verify_submitted:
                        curr_l = get_login_lock_status(target_email)
                        curr_r = get_recovery_status(target_email)
                        if curr_l["is_blocked"] or curr_l["is_locked"] or curr_r["is_locked"]:
                            st.error("Action not permitted while recovery is locked.")
                        elif not otp_input or len(otp_input) != 6:
                            st.error("Please enter a valid 6-digit numerical code.")
                        else:
                            ok, msg = verify_otp_token(st.session_state.otp_token, otp_input, target_email, JWT_SECRET, OTP_EXPIRY_MINUTES)
                            if ok:
                                reset_recovery_attempts(target_email)
                                curr_sid = st.session_state.get("rec_sid") or secrets.token_hex(16)
                                # RULE (Bug 2): Commit step='reset' and otp_verified=1 to DB and session before rerun
                                save_recovery_session(curr_sid, target_email, "otp", "reset", otp_token=None, otp_verified=True)
                                safe_set_cookie("fq_rec_sid", curr_sid, key="set_sid_reset", expires_at=datetime.datetime.now() + datetime.timedelta(hours=1))
                                st.session_state.rec_sid = curr_sid
                                st.session_state.otp_verified = True
                                st.session_state.otp_token = None
                                st.session_state.otp_stage = "reset"
                                st.rerun()
                            else:
                                record_recovery_failure(target_email)
                                new_r = get_recovery_status(target_email)
                                if new_r["is_locked"]:
                                    curr_sid = st.session_state.get("rec_sid")
                                    if curr_sid:
                                        delete_recovery_session(curr_sid)
                                    st.session_state.otp_token = None
                                    st.rerun()
                                else:
                                    attempts_left = OTP_FREE_ATTEMPTS - new_r["failed_attempts"]
                                    if attempts_left > 0:
                                        st.error(f"{msg} ({attempts_left} attempt{'s' if attempts_left > 1 else ''} remaining)")
                                    else:
                                        st.error("Maximum OTP attempts reached. This code has been invalidated.")
                                        curr_sid = st.session_state.get("rec_sid")
                                        if curr_sid:
                                            delete_recovery_session(curr_sid)
                                        st.session_state.otp_token = None
                                        st.rerun()

                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Cancel Recovery", use_container_width=True):
                        clear_reset_state()
                        navigate("Login")

                # Set New Password after verification
                elif st.session_state.otp_stage == "reset":
                    curr_sid = st.session_state.get("rec_sid")
                    db_sess = get_recovery_session(curr_sid) if curr_sid else None
                    if not (st.session_state.get("otp_verified") or (db_sess and db_sess.get("otp_verified"))):
                        st.error("Unauthorized access or verification expired. Please verify your OTP code again.")
                        clear_reset_state()
                        navigate("Forgot")

                    with st.form("new_password_form", border=False):
                        npw = st.text_input("New Password", type="password", placeholder="Min. 8 chars (upper, lower, digit, symbol)")
                        confirm_npw = st.text_input("Confirm New Password", type="password", placeholder="Re-enter new password")
                        st.markdown("<br>", unsafe_allow_html=True)

                        c1, c2 = st.columns([1.2, 1])
                        update_pwd_submitted = c1.form_submit_button("Update Password", use_container_width=True)
                        cancel_pwd_clicked = c2.form_submit_button("Cancel", use_container_width=True)

                    if cancel_pwd_clicked:
                        clear_reset_state()
                        navigate("Login")

                    elif update_pwd_submitted:
                        is_valid_pwd, pwd_msg = validate_password(npw)
                        if not is_valid_pwd:
                            st.error(f"Password Error: {pwd_msg}")
                        elif npw != confirm_npw:
                            st.error("Passwords do not match.")
                        else:
                            conn = get_db()
                            try:
                                if check_password_reused(conn, target_email, npw):
                                    st.error("For security, you cannot reuse your current or recent password. Please choose a new password.")
                                else:
                                    new_h = hash_txt(npw)
                                    conn.execute("UPDATE users SET password_hash=? WHERE email=?", (new_h, target_email))
                                    record_password_history(conn, target_email, new_h)
                                    conn.commit()
                                    reset_recovery_attempts(target_email)
                                    clear_reset_state()
                                    st.session_state.login_info_msg = "Password updated successfully. Please log in with your new password."
                                    time.sleep(1)
                                    navigate("Login")
                            finally:
                                conn.close()

                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Cancel Recovery", use_container_width=True):
                        clear_reset_state()
                        navigate("Login")

# PROTECTED DASHBOARDS
else:
    user_row = query_db("SELECT username, role FROM users WHERE email=?", (auth_email,), one=True)
    if not user_row:
        safe_delete_cookie("fq_jwt", key="del_jwt_notfound")
        st.session_state.token = None
        st.session_state.page = "Login"
        st.query_params["page"] = "Login"
        st.rerun()

    uname, role = user_row[0], user_row[1] if user_row[1] else "user"
    is_admin = (role == "admin" or auth_email.lower() == ADMIN_EMAIL.lower())

    # SIDEBAR NAVIGATION (RESTORED EXACTLY AS ORIGINAL)
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">FreightQuote AI</div>
            <div style="font-size:12px;color:{COLORS['text_muted']};margin-top:2px;">
                {"Admin Control Panel" if is_admin else "User Portal"}
            </div>
        </div>
        <hr style="border:none;border-top:1px solid {COLORS['border']};margin:8px 0 16px 0;">
        """, unsafe_allow_html=True)

        menu = option_menu(
            None, ["Dashboard", "Logout"],
            icons=["house", "box-arrow-right"],
            key="main_sidebar_menu",
            styles={
                "container": {"background-color": COLORS['bg_sidebar'], "padding": "0"},
                "nav-link": {"font-size": "14px", "color": COLORS['text_main'], "border-radius": "6px"},
                "nav-link-selected": {"background-color": COLORS['accent_hover'], "color": COLORS['text_heading'], "font-weight": "600"}
            }
        )

        if menu == "Logout":
            safe_delete_cookie("fq_jwt", key="del_jwt_logout")
            st.session_state.token = None
            st.session_state.page = "Login"
            st.query_params["page"] = "Login"
            st.session_state.pop("main_sidebar_menu", None)
            clear_reset_state()
            time.sleep(0.3)
            st.rerun()

    # ADMIN DASHBOARD
    if is_admin:
        st.markdown(f"""
        <div style="background:{COLORS['bg_card']};border:1px solid {COLORS['border']};border-radius:10px;padding:20px 24px;margin-bottom:24px;display:flex;justify-content:space-between;align-items:center;">
            <div>
                <h2 style="margin:0;font-size:20px;color:{COLORS['text_heading']};">FreightQuote AI — Admin Dashboard</h2>
                <div style="color:{COLORS['text_muted']};font-size:13px;margin-top:4px;">Logged in as: {uname}</div>
            </div>
            <div style="font-size:12px;background:#F0FDF4;color:#16A34A;padding:6px 14px;border-radius:20px;font-weight:600;border:1px solid #BBF7D0;">
                Administrator Role
            </div>
        </div>
        """, unsafe_allow_html=True)

        users_list = query_db("SELECT id, username, email, role, created_at FROM users ORDER BY id ASC")
        now_ts = time.time()

        table_data = []
        for u in users_list:
            u_id, u_uname, u_email, u_role, u_created = u
            u_key = u_email.lower().strip()
            l_stat = get_login_lock_status(u_key)
            r_stat = get_recovery_status(u_key)

            if l_stat["is_blocked"]:
                status_str = "Blocked"
            elif l_stat["is_locked"]:
                status_str = f"Locked ({l_stat['remaining_seconds']}s)"
            elif r_stat["is_locked"]:
                status_str = f"Locked ({r_stat['remaining_seconds']}s)"
            else:
                status_str = "Active"

            table_data.append({
                "User ID": u_id,
                "Username": u_uname,
                "Email Address": u_email,
                "Role": u_role if u_role else "user",
                "Status": status_str,
                "Registered At": u_created
            })

        st.markdown("### Registered Users")
        import pandas as pd
        df = pd.DataFrame(table_data)
        st.dataframe(df, use_container_width=True, hide_index=True)

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown("### User Account Management")
        unblockable_users = [u for u in table_data if u["Status"] != "Active"]
        if unblockable_users:
            options_dict = {f"{u['Username']} ({u['Email Address']}) - {u['Status']}": u['Email Address'] for u in unblockable_users}
            selected_label = st.selectbox("Select account to unlock or unblock", list(options_dict.keys()))
            if st.button("Unlock / Unblock Account", use_container_width=True):
                target_to_unblock = options_dict[selected_label]
                admin_unblock_user(target_to_unblock)
                st.success(f"Account for {target_to_unblock} has been successfully unlocked/unblocked.")
                time.sleep(1)
                st.rerun()
        else:
            st.info("All user accounts are currently Active. No accounts are locked or blocked.")

    # USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['bg_card']};border:1px solid {COLORS['border']};border-radius:10px;padding:20px 24px;margin-bottom:24px;">
            <h2 style="margin:0;font-size:20px;color:{COLORS['text_heading']};">Welcome to FreightQuote AI</h2>
            <div style="color:{COLORS['text_muted']};font-size:13px;margin-top:4px;">Intelligent Freight Rate Estimation & Logistics Optimization</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("### Account Overview")
        st.markdown(f"""
        <div class="pn-card">
            <div class="account-row">
                <span class="account-label">Username</span>
                <span class="account-value">{uname}</span>
            </div>
            <div class="account-row">
                <span class="account-label">Email Address</span>
                <span class="account-value">{auth_email}</span>
            </div>
            <div class="account-row">
                <span class="account-label">Role</span>
                <span class="account-value">{role}</span>
            </div>
        </div>
        """, unsafe_allow_html=True)


Overwriting app.py


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok

# 1. Safely retrieve ngrok authentication token from Colab Secrets
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
    colab_env = True
except Exception:
    NGROK_TOKEN = os.getenv("NGROK_AUTHTOKEN")
    userdata = None
    colab_env = False

if NGROK_TOKEN:
    from pyngrok import conf
    conf.get_default().auth_token = NGROK_TOKEN
else:
    print(" NGROK_AUTHTOKEN not found in Colab Secrets. Add 'NGROK_AUTHTOKEN' to your Colab Secrets tab.")

# 2. Kill any existing ngrok tunnels or previous streamlit sessions cleanly
ngrok.kill()
!pkill -f streamlit
time.sleep(2)

# 3. Propagate Colab Secrets safely to the Streamlit subprocess environment
sub_env = os.environ.copy()
for secret_key in ["EMAIL_PASSWORD", "JWT_SECRET", "EMAIL_ID", "SENDER_EMAIL", "ADMIN_PASSWORD", "ADMIN_EMAIL"]:
    try:
        val = userdata.get(secret_key) if userdata else os.getenv(secret_key)
        if val:
            sub_env[secret_key] = val
    except Exception:
        pass

# 4. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=sub_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# 5. Connect and expose public ngrok tunnel
try:
    public_url = ngrok.connect(8501).public_url

    print(f"FreightQuote AI Live URL: {public_url}")
    print("\n")
    print("App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

    while True:
        time.sleep(1)

except KeyboardInterrupt:
    print("\n")
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("\n")
    print("Ngrok tunnel closed and Streamlit server stopped gracefully.")


FreightQuote AI Live URL: https://mandatory-fantasize-finished.ngrok-free.dev


App is running! Press [Ctrl + C] or the Colab Stop button to shut down.
